# MLflow autologging

Writing manual logging code for every parameter and metric can get tedious. MLflow's **autologging** feature handles most of this for you by hooking into common libraries like scikit-learn and XGBoost. In this notebook, we'll cover:

1. Setting up autologging for scikit-learn and XGBoost.
2. Adding your own custom metrics on top of what's logged automatically.
3. Using autologging with nested runs for hyperparameter tuning.
4. Best practices for when to use autologging versus manual logging.

## 1. Setup and imports

In [1]:
import sys, os

REPO_ROOT = os.path.abspath(os.pardir)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

from src.data_preprocessing import load_data, preprocess, split_data

print("MLflow version:", mlflow.__version__)
mlflow.set_tracking_uri(f"sqlite:///{REPO_ROOT}/mlflow.db")

# Start clean to avoid surprises from previous notebook state
mlflow.autolog(disable=True)

MLflow version: 3.15.2


## 2. Load data and define experiment

In [2]:
EXPERIMENT_NAME = "telco-churn-autolog"
mlflow.set_experiment(EXPERIMENT_NAME)

df = load_data()
X, y, scaler, feature_names = preprocess(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

2026/09/02 13:42:05 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-autolog' does not exist. Creating a new experiment.


Experiment: telco-churn-autolog
Train: (80000, 30), Test: (20000, 30)


## 3. Scikit-learn autologging

`mlflow.sklearn.autolog()` is great because it automatically grabs your hyperparameters, metrics, and even the model itself when you call `.fit()`. 

We'll still log some custom metric manually here just to show how you can combine both approaches.

In [3]:
mlflow.sklearn.autolog(log_model_signatures=True, log_input_examples=True)

with mlflow.start_run(run_name="autolog-rf-flat"):
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("tracking_style", "autolog_flat")

    rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    proba = rf.predict_proba(X_test)[:, 1]

    # Custom metrics (not automatic unless computed by your code)
    mlflow.log_metric("custom_f1", f1_score(y_test, preds))
    mlflow.log_metric("custom_recall", recall_score(y_test, preds))
    mlflow.log_metric("custom_roc_auc", roc_auc_score(y_test, proba))

    print("RandomForest autolog run created.")

mlflow.sklearn.autolog(disable=True)

2026/09/02 13:42:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest autolog run created.


## 4. XGBoost autologging

Autologging works pretty much the same way for XGBoost. It'll capture framework-specific details, though the exact metrics and artifacts might look a bit different than scikit-learn.

In [4]:
mlflow.xgboost.autolog(log_model_signatures=True, log_input_examples=True)

with mlflow.start_run(run_name="autolog-xgb-flat"):
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("tracking_style", "autolog_flat")

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
    )
    xgb.fit(X_train, y_train)

    preds = xgb.predict(X_test)
    proba = xgb.predict_proba(X_test)[:, 1]

    # Additional custom metrics
    mlflow.log_metric("custom_f1", f1_score(y_test, preds))
    mlflow.log_metric("custom_recall", recall_score(y_test, preds))
    mlflow.log_metric("custom_roc_auc", roc_auc_score(y_test, proba))

    print("XGBoost autolog run created.")

mlflow.xgboost.autolog(disable=True)

2026/09/02 13:44:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost autolog run created.


## 5. Nested runs and autologging

This is where autologging is most useful. When you're running a big grid search, autologging ensures every single trial is recorded in detail without you having to write the logging code inside the loop.

In [5]:
from itertools import product

mlflow.xgboost.autolog(log_model_signatures=True, log_input_examples=False)

grid_n_estimators = [100, 200]
grid_max_depth = [4, 8]
grid_lr = [0.05, 0.1]

best_trial = {"f1": 0.0, "params": None}

with mlflow.start_run(run_name="autolog-xgb-grid-parent"):
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("tracking_style", "autolog_nested_tuning")

    for n_est, depth, lr in product(grid_n_estimators, grid_max_depth, grid_lr):
        trial_name = f"xgb-n{n_est}-d{depth}-lr{lr}"

        with mlflow.start_run(run_name=trial_name, nested=True):
            model = XGBClassifier(
                n_estimators=n_est,
                max_depth=depth,
                learning_rate=lr,
                eval_metric="logloss",
                random_state=42,
            )
            model.fit(X_train, y_train)

            preds = model.predict(X_test)
            trial_f1 = f1_score(y_test, preds)
            mlflow.log_metric("custom_f1", trial_f1)

            if trial_f1 > best_trial["f1"]:
                best_trial["f1"] = trial_f1
                best_trial["params"] = {
                    "n_estimators": n_est,
                    "max_depth": depth,
                    "learning_rate": lr,
                }

    # Parent summary metrics/params
    mlflow.log_metric("best_custom_f1", best_trial["f1"])
    mlflow.log_params({f"best_{k}": v for k, v in best_trial["params"].items()})

print("Nested autolog tuning run created.")
mlflow.xgboost.autolog(disable=True)

2026/09/02 13:45:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:45:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:45:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:45:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:45:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:46:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 13:46:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Nested autolog tuning run created.


## 6. Comparing runs with code

As a reminder, you can always query and compare your runs using the Python API. It doesn't matter if they were logged manually or via autologging.

In [6]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.custom_f1 DESC", "metrics.best_custom_f1 DESC"],
)

cols = [
    "run_id",
    "tags.mlflow.runName",
    "tags.model_type",
    "tags.tracking_style",
    "tags.mlflow.parentRunId",
    "params.n_estimators",
    "params.max_depth",
    "params.learning_rate",
    "metrics.custom_f1",
    "metrics.best_custom_f1",
]

runs_df[[c for c in cols if c in runs_df.columns]].head(20)

,run_id,tags.mlflow.runName,tags.model_type,tags.tracking_style,tags.mlflow.parentRunId,params.n_estimators,params.max_depth,params.learning_rate,metrics.custom_f1,metrics.best_custom_f1
0,6c201c4b748644798dd0b82ceb54a287,xgb-n200-d4-lr0.1,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,4,0.1,0.722795,NaN
1,b5db1612c60b440a861f0abbc1472a92,xgb-n100-d4-lr0.05,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,4,0.05,0.722535,NaN
2,0f790700bdc1410da75e9b75ed6f14d5,xgb-n100-d4-lr0.1,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,4,0.1,0.722184,NaN
3,f440ded54fc340ce82523d0e554e248d,xgb-n200-d4-lr0.05,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,4,0.05,0.722181,NaN
4,3611f4eafd9144b295d7f08cf0664cc2,autolog-xgb-flat,XGBoost,autolog_flat,None,None,6,0.05,0.721649,NaN
5,6c6d929c20194b79aef4c9354632079b,autolog-rf-flat,RandomForest,autolog_flat,None,200,15,None,0.720611,NaN
6,6309ab6879424cb38586434bb2a54ad7,xgb-n200-d8-lr0.05,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,8,0.05,0.719809,NaN
7,d9449d24ef554d41bd0220b76620ac49,xgb-n100-d8-lr0.05,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,8,0.05,0.719478,NaN
8,59b80b69111848ef812da40883913eff,xgb-n100-d8-lr0.1,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,8,0.1,0.719085,NaN
9,9f72eb38befc4d589c03ec1373f361b8,xgb-n200-d8-lr0.1,None,None,6fa7cab3c47c4fe3aaea9318816f0d6b,None,8,0.1,0.716685,NaN


## 7. What actually gets captured?

Autologging usually grabs:

- **Parameters**: Everything you passed to the model constructor.
- **Metrics**: Standard scores like accuracy or log-loss.
- **Artifacts**: The model file, requirement files, and environment metadata.
- **Signatures**: The expected input and output types for your model.

Let's peek at a specific run to see what MLflow saved.

In [7]:
from mlflow import MlflowClient

client = MlflowClient()

# Pick first autolog run
example_run_id = runs_df.iloc[0]["run_id"]
run = client.get_run(example_run_id)

print("Run ID:", example_run_id)
print("\nTop tags:")
for k, v in sorted(run.data.tags.items())[:12]:
    print(f"  {k}: {v}")

print("\nTop metrics:")
for k, v in sorted(run.data.metrics.items())[:12]:
    print(f"  {k}: {v}")

Run ID: 6c201c4b748644798dd0b82ceb54a287

Top tags:
  mlflow.parentRunId: 6fa7cab3c47c4fe3aaea9318816f0d6b
  mlflow.runName: xgb-n200-d4-lr0.1
  mlflow.source.name: 04_autologging.ipynb
  mlflow.source.type: NOTEBOOK
  mlflow.user: vagrant

Top metrics:
  custom_f1: 0.7227954565160842


## 8. Tips and best practices

- **Use flat runs** for simple, one-off model comparisons.
- **Use nested runs** when you're doing hyperparameter tuning or any kind of iterative search.
- **Keep manual logging** for the metrics that actually matter to your business (like custom recall or cost-based metrics).
- **Use clear tags** (like `tracking_style` or `model_type`) to keep your experiments organized.
- **Turn off autologging** when you're done with a section to avoid accidentally logging things you didn't mean to.

### Launching the UI

```bash
mlflow ui --port 5000
```

Open [http://127.0.0.1:5000](http://127.0.0.1:5000) and check out the `telco-churn-autolog` experiment.